# Step 1 — PDF → structured paragraph nodes

**Goal.** Turn every Certivity-rendered UN Regulation PDF (56 base + 36 latest) into a flat list of
*paragraph nodes*, because the golden answers are addressed by `(regulation, version, paragraph, page)`.

**What the PDFs give us**
- clean text layer; every page starts with `UN R148, version 1.6.0 | page 11 of 48`
- paragraphs start with a label at line start: `6.10.4.2. The rear position lamps shall …`
- annexes (and Parts / Appendices inside them) restart numbering → a node's address is `(doc, label)`,
  e.g. `Annex 4 Part 1 1.2`
- noise: wrapped lines, tables flattened one cell per line, footnotes, references that look like headings

**Node** = `{reg, version, role, doc, label, key, depth, parent, page_start, page_end, heading, title, text, tables, has_shall}`
where `key` uses the exact normalisation of `tools/score_local.py`, so our keys match the scorer's by construction.

**Rules**
1. page pass: PyMuPDF4LLM renders each page to Markdown (tables become `| grid |` rows, cached in `work/md/`); we strip its heading/bold decoration, parse the header → page number + version, drop it
2. a line starts a node if it matches `^N(.N)*.? Text` (dot optional below depth 1, text must start upper-case)
3. a new label is accepted only if its parent *or grandparent* already exists in the current doc, or it is the next
   top-level number, or nothing is open yet (self-healing: one missed label does not swallow the rest of the file)
4. `Annex N [title]`, `Part N`, `Appendix N` switch the numbering space; bare `Annex N` mid-page (table cell) does not
5. short, title-like nodes become headings; the nearest heading titles are carried down as `heading`

**Check.** All 190 golden rear-lamp citations must resolve to a node whose page matches (±1).

In [1]:
from pathlib import Path
import json, re, sys, time, collections, warnings
warnings.filterwarnings("ignore")

ROOT   = Path.cwd() if (Path.cwd() / "package").exists() else Path.home() / "Desktop/certivity"
PKG    = ROOT / "package"
PDFS   = PKG / "regulations/pdf"
PARSED = ROOT / "work/parsed"
PARSED.mkdir(parents=True, exist_ok=True)
INDEX  = json.loads((PKG / "regulations/index.json").read_text())
print(ROOT, "|", len(INDEX), "regulations in index")

/Users/arnavsingh/Desktop/certivity | 56 regulations in index


In [2]:
%%writefile work/regparse.py
"""Parse a Certivity-rendered UN Regulation PDF into a flat list of paragraph nodes (generated from the notebook)."""
from __future__ import annotations
import json, re, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
warnings.filterwarnings("ignore")
import pymupdf, pymupdf4llm

MD_CACHE = Path(__file__).resolve().parent / "md"   # per-file cache of the PyMuPDF4LLM markdown

HEADER_RE   = re.compile(r"^(UN (?:R\d+[A-Z]?|GTR \d+)), version (\S+) \| page (\d+) of (\d+)\s*$")
LABEL_RE    = re.compile(r"^(\d+(?:\.\d+)*)(\.)?\s+([A-Z(\"“'].*)$")      # "6.10.4.2. Text" / "5.7 Grouped"
ANNEX_RE    = re.compile(r"^Annex\s+(\d+[A-Z]?)\s*(?:[-–:]\s*)?(.*)$", re.I)
PART_RE     = re.compile(r"^(Part|Appendix)\s+(\d+[A-Z]?)\b\s*(?:[-–:]\s*)?(.*)$", re.I)
TABLE_RE    = re.compile(r"^Table\s+([A-Z]?\d+[A-Z]?(?:-\d+)?)\b")


def paragraph_key(label: str) -> str:
    """Same normalisation as tools/score_local.py so our keys match the scorer's."""
    text = str(label or "").casefold().replace("§", " ")
    text = re.sub(r"[,;:()\[\]]", " ", text)
    text = re.sub(r"(?<![0-9a-z])\.|\.(?![0-9a-z])", " ", text)
    text = " ".join(text.split())
    return re.sub(r"^(paragraphs?|para|par|section|clause)\s+", "", text)


@dataclass
class Node:
    reg: str; version: str; role: str
    doc: str            # "main" | "Annex 3" | "Annex 4 Part 1" | "Annex 9 Appendix 1"
    label: str          # "6.10.4.2"
    key: str            # normalised full address: "annex 4 part 1 1.2" / "6.10.4.2"
    depth: int
    parent: str | None
    page_start: int; page_end: int
    heading: str        # titles of the enclosing sections, outermost first ("Individual specifications > Rear position lamp")
    title: str | None   # this node's own title if it is a heading
    text: str
    tables: list[str] = field(default_factory=list)
    has_shall: bool = False


def full_label(doc: str, label: str) -> str:
    return label if doc == "main" else f"{doc} {label}"


def _clean_md_line(line: str) -> str:
    """Strip the markdown decoration PyMuPDF4LLM adds (heading marks, bold/italic at line start,
    <sup>/<br> tags) so a paragraph label is again at the very start of the line."""
    line = line.rstrip()
    if line.lstrip().startswith("|"):                                   # table row: keep the grid, tidy the cells
        line = re.sub(r"<sup>(.*?)</sup>", r"^\1", line)
        line = line.replace("<br>", " ").replace("**", "").replace("_", "")
        return re.sub(r"\s+", " ", line).strip()
    line = re.sub(r"^\s*#+\s*", "", line)
    line = re.sub(r"^\s*[-•*]\s+", "", line)                              # bullet that PyMuPDF4LLM put in front of a label
    line = re.sub(r"^\s*[*_]{1,3}(.*?)[*_]{1,3}(?=\s|$)", r"\1", line)
    line = re.sub(r"<sup>(.*?)</sup>", r"\1", line).replace("<br>", " ")
    return line.strip()


def page_markdown(pdf_path):
    """[(page_no, markdown)] for the file, from PyMuPDF4LLM, cached on disk (the conversion takes ~8 s per 100 pages)."""
    pdf_path = Path(pdf_path)
    MD_CACHE.mkdir(parents=True, exist_ok=True)
    cache = MD_CACHE / (pdf_path.parent.parent.name + "_" + pdf_path.parent.name + ".json")
    if cache.exists():
        return [tuple(x) for x in json.loads(cache.read_text())]
    chunks = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True, show_progress=False)
    out = [(i + 1, c["text"]) for i, c in enumerate(chunks)]
    cache.write_text(json.dumps(out, ensure_ascii=False))
    return out


def extract_pages(pdf_path):
    """-> (regulation, version, [(page_no, cleaned_lines_without_header), ...])"""
    reg = version = None; pages = []
    for pdf_page_no, md in page_markdown(pdf_path):
        lines, page_no = [], pdf_page_no
        for raw in md.splitlines():
            s = raw.strip()
            if (m := HEADER_RE.match(s)):
                reg, version, page_no = m.group(1), m.group(2), int(m.group(3)); continue
            s = _clean_md_line(raw)
            if s and s != "-----":
                lines.append(s)
        pages.append((page_no, lines))
    if reg is None:
        raise ValueError(f"{pdf_path}: no Certivity page header found")
    return reg, version, pages


def _is_annex_heading(line: str, first_on_page: bool):
    m = ANNEX_RE.match(line)
    if not m or len(line) > 170 or line.endswith("."):
        return None
    title = (m.group(2) or "").strip()
    if title and not re.match(r"^[A-Z(\"“]", title) and not title.lower().startswith("appendix"):
        return None                                   # "Annex 4 to this Regulation" -> a reference, not a heading
    if (not title or title.startswith("(")) and not first_on_page:
        return None                                   # bare "Annex 5" or "Annex 10 (values in" inside a table
    return f"Annex {m.group(1)}", title


def parse_pdf(pdf_path, role: str) -> list[Node]:
    reg, version, pages = extract_pages(pdf_path)
    nodes: list[Node] = []
    doc, annex = "main", None
    cur: Node | None = None
    buf: list[str] = []
    headings: dict[int, str] = {}                     # depth -> title within the current doc
    in_doc: set[str] = set()                          # labels already created in the current doc
    top_max = 0                                       # highest depth-1 number seen in the current doc

    def flush():
        nonlocal cur, buf
        if cur is None:
            return
        text = ""
        for s in (s.strip() for s in buf if s.strip()):          # table rows keep their own line
            text += ("\n" if (s.startswith("|") or text.endswith("|")) else " ") + s if text else s
        cur.text = text
        cur.has_shall = bool(re.search(r"\bshall\b", text))
        cur.tables = sorted({m.group(1) for s in buf if (m := TABLE_RE.match(s.strip()))})
        if cur.depth <= 3 and len(text) <= 90 and not text.endswith((".", ";", ":")) and not cur.has_shall:
            cur.title = text
            headings[cur.depth] = text
            for d in [d for d in headings if d > cur.depth]:
                del headings[d]
        cur.heading = " > ".join(headings[d] for d in sorted(headings) if d < cur.depth)
        nodes.append(cur); cur, buf = None, []

    def switch(new_doc: str, title: str):
        nonlocal doc, headings, in_doc, top_max
        flush(); doc = new_doc; headings = {0: f"{new_doc} {title}".strip()}; in_doc = set(); top_max = 0

    for page_no, lines in pages:
        first = True
        for raw in lines:
            line = raw.strip()
            if not line:
                continue
            if (ah := _is_annex_heading(line, first)):
                annex = ah[0]
                am = re.match(r"^(.*?)\s*(?:[-–:]\s*)?(Appendix\s+\d+[A-Z]?)\s*(?:[-–:]\s*)?(.*)$", ah[1], re.I) if "appendix" in ah[1].lower() else None
                if am:   # "Annex 9 - Appendix 1 BCI test"
                    switch(f"{annex} {am.group(2).title()}", am.group(3))
                else:
                    switch(annex, ah[1])
                first = False; continue
            pm = PART_RE.match(line)
            if pm and annex and len(line) <= 170 and not line.endswith(".") and (not pm.group(3) or re.match(r"^[A-Z(\"“]", pm.group(3))):
                switch(f"{annex} {pm.group(1).title()} {pm.group(2)}", pm.group(3)); first = False; continue
            first = False
            lm = LABEL_RE.match(line)
            if lm:
                label, rest = lm.group(1), lm.group(3)
                parts = label.split("."); depth = len(parts)
                parent = ".".join(parts[:-1]) or None
                grand  = ".".join(parts[:-2]) or None
                ok = (depth == 1 and lm.group(2) == "." and int(label) in (top_max + 1, top_max + 2, 0, 1) and label not in in_doc) \
                     or (depth >= 2 and (parent in in_doc or (grand and grand in in_doc)) and label not in in_doc) \
                     or (cur is None and not nodes)
                if ok:
                    flush()
                    cur = Node(reg=reg, version=version, role=role, doc=doc, label=label,
                               key=paragraph_key(full_label(doc, label)), depth=depth, parent=parent,
                               page_start=page_no, page_end=page_no, heading="", title=None, text="")
                    buf = [rest]; in_doc.add(label)
                    if depth == 1: top_max = max(top_max, int(label))
                    continue
            if cur is None:                           # preamble / text before the first label of this doc
                cur = Node(reg=reg, version=version, role=role, doc=doc, label="0", key=paragraph_key(full_label(doc, "0")),
                           depth=1, parent=None, page_start=page_no, page_end=page_no, heading="", title=None, text="")
                buf = []
            buf.append(line); cur.page_end = page_no
    flush()
    return nodes


def save(nodes, out_path):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    Path(out_path).write_text(json.dumps([asdict(n) for n in nodes], ensure_ascii=False, indent=1))

def load(path):
    return [Node(**d) for d in json.loads(Path(path).read_text())]


class Corpus:
    """All parsed regulations in memory. find / toc / read_section are the future agent tools."""
    def __init__(self, parsed_dir):
        self.by_reg, self.index = {}, {}
        for p in sorted(Path(parsed_dir).glob("*.json")):
            nodes = load(p)
            if nodes:
                self.by_reg[(nodes[0].reg, nodes[0].role)] = nodes
                for n in nodes:
                    self.index.setdefault((n.reg, n.role, n.key), n)

    def find(self, reg, label, role="base"):
        key = paragraph_key(label)
        n = self.index.get((reg, role, key))
        if n is None:                                  # golden labels may append a table: "5.1.4.4.1. Table 3"
            n = self.index.get((reg, role, re.sub(r"\s+table\s+\S+$", "", key)))
        return n

    def toc(self, reg, role="base", max_depth=2):
        return [n for n in self.by_reg[(reg, role)] if n.depth <= max_depth and (n.title or n.depth == 1)]

    def read_section(self, reg, label, role="base"):
        key = paragraph_key(label)
        return [n for n in self.by_reg[(reg, role)] if n.key == key or n.key.startswith(key + ".")]


Overwriting work/regparse.py


In [3]:
import importlib, sys
sys.path.insert(0, str(ROOT / "work"))
import regparse as rp; importlib.reload(rp)

t = time.time(); rows = []
for e in INDEX:
    slug = "_".join(e["regulation"].split())
    for v in e["versions"]:
        for f in v["files"]:
            nodes = rp.parse_pdf(PDFS / slug / v["role"] / f["name"], v["role"])
            rp.save(nodes, PARSED / f"{slug}_{v['role']}.json")
            rows.append(dict(reg=e["regulation"], role=v["role"], version=v["version_label"], pages=max(n.page_end for n in nodes),
                             nodes=len(nodes), annex_nodes=sum(n.doc != "main" for n in nodes), shall=sum(n.has_shall for n in nodes)))
print(f"parsed {len(rows)} files in {time.time()-t:.1f}s")
import pandas as pd
pd.DataFrame(rows)

parsed 92 files in 0.7s


,reg,role,version,pages,nodes,annex_nodes,shall
0,UN R3,base,2.17.0,26,181,114,98
1,UN R6,base,2.0.0,25,210,74,117
2,UN R7,base,3.0.0,28,215,67,112
3,UN R10,base,7.2.0,147,683,391,324
4,UN R10,latest,7.3.0,149,691,391,326
...,...,...,...,...,...,...,...
87,UN R174,base,1.0.1,13,139,33,57
88,UN R174,latest,1.1.0,14,169,46,61
89,UN R175,base,0.1.0,17,197,86,95
90,UN R175,latest,1.0.0,19,231,86,111


In [4]:
# --- validation against the golden rear-lamp answer: every citation must resolve to a node on the right page ---
C = rp.Corpus(PARSED)
exp = json.loads((PKG / "cases/dev-rear-lamp/expected.json").read_text())["requirements"]
ok, misses = 0, []
for r in exp:
    s = r["source"]; n = C.find(s["regulation"], s["paragraph"])
    if n is None:
        misses.append((s["regulation"], s["paragraph"], s["page"], "UNRESOLVED"))
    elif not (n.page_start - 1 <= s["page"] <= n.page_end + 1):
        misses.append((s["regulation"], s["paragraph"], s["page"], f"page mismatch: node p{n.page_start}-{n.page_end}"))
    else:
        ok += 1
print(f"golden citations resolved with correct page: {ok}/{len(exp)}")
for m in misses: print("  ", m)

golden citations resolved with correct page: 190/190


In [5]:
# --- the future agent tools, on the rear-lamp regulations ---
for n in C.toc("UN R48", max_depth=2)[:40]:
    print(f"p{n.page_start:<4} {n.doc:<10} {n.label:<8} {n.title or n.text[:60]}")

p1    main       0        UN R48 (version 9.3.0) Consolidated text of this version, re
p1    main       1        Scope This Regulation applies to vehicles of categories M, N
p1    main       2        Definitions
p1    main       2.1      General
p1    main       2.2      Type definitions
p2    main       2.3      Vehicle
p3    main       2.4      Lamps generalities
p5    main       2.5      Lamps
p6    main       2.6      Signal
p6    main       2.7      System
p8    main       2.8      Lens
p8    main       2.9      Light sources
p9    main       2.10     Photometry
p12   main       2.11     Colour
p17   main       3        Application for approval
p18   main       4        Approval
p19   main       5        General specifications
p19   main       5.7      Grouped, combined or reciprocally incorporated or single lamps
p22   main       5.14     Concealable lamps
p24   main       5.16     Number of lamps
p27   main       5.28     General provisions relating to geometric visibility
p30  

In [6]:
sec = C.read_section("UN R48", "6.10")
print(len(sec), "nodes in R48 §6.10  (pages", sec[0].page_start, "-", sec[-1].page_end, ")\n")
for n in sec[:12]:
    print(f"[p{n.page_start} §{n.label}] ({n.heading})\n    {n.text[:160]}")
print("\nfind R148 'Annex 3 1.2' ->", (n := C.find("UN R148", "Annex 3 1.2")) and (n.page_start, n.text[:120]))

16 nodes in R48 §6.10  (pages 50 - 51 )

[p50 §6.10] (Individual specifications)
    Rear position lamp
[p50 §6.10.1] (Individual specifications > Rear position lamp)
    Presence Devices of R or R1 or R2 categories: Mandatory
[p50 §6.10.2] (Individual specifications > Rear position lamp)
    Number Two, type-approved according to the 02 or subsequent series of amendments to UN Regulation No. 7, or to the 00 or subsequent series of amendments to UN R
[p50 §6.10.2.1] (Individual specifications > Rear position lamp > Presence Devices of R or R1 or R2 categories: Mandatory)
    Except the case where end-outline marker lamps are installed, two optional position lamps may be installed on all vehicles in categories M2, M3, N2, N3, O2, O3,
[p50 §6.10.3] (Individual specifications > Rear position lamp)
    Arrangement No special requirement.
[p50 §6.10.4] (Individual specifications > Rear position lamp)
    Position
[p50 §6.10.4.1] (Individual specifications > Rear position lamp > Position)
  

In [7]:
# --- a node that carries a table: R150 §5.1.4.4.1 (Table 3, cited by the golden set) ---
n = C.find("UN R150", "5.1.4.4.1")
print(f"[p{n.page_start}-{n.page_end} §{n.label}] tables={n.tables}\n")
print(n.text[:1800])

[p12-13 §5.1.4.4.1] tables=['3']

When measured as described in paragraph 3 of Part 1 of Annex 4, the RI values for red retro-reflectors must be equal to or greater than those in Table 3, expressed in millicandelas per lux, for the angles of divergence and illumination shown. Table 3 **Requirements for RI values of Retro-reflectors**
|||||Angles|in degrees|||||
|---|---|---|---|---|---|---|---|---|---|
|Class|Colour|Illumination angles [º]|Vertical β1|0°|±10°|±5°|0°|0°|0°|
||||Horizontal β2|0° Minim|0° um require|±20° ments for|±30° RI values i|±40° n mcd·lx^-1|±50°|
|IA, IB|White|Angle of divergence α|20'|1.20∙10^3|8.00∙10^2|4.00∙10^2|--|--|--|
|---|---|---|---|---|---|---|---|---|---|
||||1°30'|2.0∙10^1|1.12∙10^1|1.0∙10^1|--|--|--|
||Amber||20'|7.50∙10^2|5.00∙10^2|2.50∙10^2|--|--|--|
||||1°30'|1.25∙10^1|7∙10^0|6.25∙10^0|--|--|--|
||Red||20'|3.00∙10^2|2.00∙10^2|1.00∙10^2|--|--|--|
||||1°30'|5∙10^0|2.8∙10^0|2.5∙10^0|--|--|--|
|IIIA, IIIB|White||20'|1.80∙10^3|8.00∙10^2|6.00∙10^2|--|--|-